In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv("../data/raw/train.csv")

In [3]:
#Handle missing values found in Postal Code 
# Which rows have missing Postal Code — inspect them to decide on an imputation strategy
df[df['Postal Code'].isnull()][['City', 'State', 'Region', 'Postal Code']]

,City,State,Region,Postal Code
2234,Burlington,Vermont,East,NaN
5274,Burlington,Vermont,East,NaN
8798,Burlington,Vermont,East,NaN
9146,Burlington,Vermont,East,NaN
9147,Burlington,Vermont,East,NaN
9148,Burlington,Vermont,East,NaN
9386,Burlington,Vermont,East,NaN
9387,Burlington,Vermont,East,NaN
9388,Burlington,Vermont,East,NaN
9389,Burlington,Vermont,East,NaN


All 11 missing Postal Codes are the same single location: Burlington, Vermont, Region = East. That's not 11 separate unknowns, it's one missing lookup value repeated 11 times.

Burlington, VT has a real, findable ZIP code (05401). Since City+State+Region are already consistent across all 11 rows, we can safely impute all of them with the correct value rather than dropping any rows

In [4]:
# 1. Impute the missing Burlington, VT rows with the correct ZIP
df.loc[(df['City'] == 'Burlington') & (df['State'] == 'Vermont'), 'Postal Code'] = 5401

# 2. Convert to int first (removes the .0 from float), then zero-pad to 5 digits as a string
df['Postal Code'] = df['Postal Code'].astype(int).astype(str).str.zfill(5) # pads extra 0 before postal code at without 0 it is only of 4 digit while rest are of 5 digits
df['Postal Code'].isnull().sum()

0

In [5]:
df['Postal Code'].str.len().value_counts() #check all postal code len is of 5 digits or not

Postal Code
5    9800
Name: count, dtype: int64

## City + State → Region consistency check

Before building `dim_location`, confirm that a City+State combination never maps to more than one Region — this determines whether City+State is a safe natural key for the dimension.

In [6]:
location_consistency = df.groupby(['City', 'State'])['Region'].nunique()
bad_locations = location_consistency[location_consistency > 1]
print(f"City+State combos mapping to more than one Region: {len(bad_locations)}")
bad_locations

City+State combos mapping to more than one Region: 0


Series([], Name: Region, dtype: int64)

**Insight (fill in after running):** If this returns 0, City+State is a safe, stable natural key for `dim_location`. If any combos show more than one Region, treat Region as an attribute that needs to be resolved (e.g., pick the most frequent Region per City+State) rather than assumed constant.

## Build `dim_product`

Recall from EDA: 32 of 1,861 `Product ID`s are reused across two genuinely different products (Category/Sub-Category matched by coincidence, Product Name did not). `Product ID` alone is therefore **not** a safe natural key — we use `Product ID + Product Name` as the true natural key, then generate a clean surrogate integer key for the warehouse.

In [7]:
# Treat (Product ID + Product Name) as the true natural key,
# since Product ID alone isn't reliably unique to one product.
df['product_natural_key'] = df['Product ID'] + '|' + df['Product Name']

print(f"Distinct products using composite key: {df['product_natural_key'].nunique()}")
# Should be roughly 1861 + 32 = 1893 (one extra per colliding ID)

Distinct products using composite key: 1893


In [8]:
dim_product = (
    df[['product_natural_key', 'Product ID', 'Product Name', 'Category', 'Sub-Category']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_product['product_key'] = dim_product.index + 1  # surrogate PK
dim_product.head()

,product_natural_key,Product ID,Product Name,Category,Sub-Category,product_key
0,FUR-BO-10001798|Bush Somerset Collection Bookcase,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases,1
1,FUR-CH-10000454|Hon Deluxe Fabric Upholstered ...,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs,2
2,OFF-LA-10000240|Self-Adhesive Address Labels f...,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels,3
3,FUR-TA-10000577|Bretford CR4500 Series Slim Re...,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,Furniture,Tables,4
4,OFF-ST-10000760|Eldon Fold 'N Roll Cart System,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,Office Supplies,Storage,5


In [9]:
# Map product_key back onto every order line so fact_sales can use it later.
# NOTE: this is the piece that was missing — without it, dim_product exists
# but nothing in df knows which surrogate key each row belongs to.
df = df.merge(
    dim_product[['product_natural_key', 'product_key']],
    on='product_natural_key',
    how='left'
)

assert df['product_key'].isnull().sum() == 0, "Some rows failed to match a product_key"
print(f"dim_product rows: {dim_product.shape[0]}")
print(f"df rows with product_key assigned: {df['product_key'].notnull().sum()} / {df.shape[0]}")

dim_product rows: 1893
df rows with product_key assigned: 9800 / 9800


## Build `dim_customer` (with SCD Type 2)

Recall from EDA: `Customer ID → Customer Name` and `Customer ID → Segment` are both true 1:1 mappings (0 inconsistencies) — so these are genuine, stable customer attributes and belong in `dim_customer`. City/State do **not** belong here (they're order-level shipping attributes — see `dim_location` below).

`Segment` is the attribute we track historically with SCD Type 2, since it's a real customer-level attribute that can plausibly change over time (e.g., a customer's account gets reclassified from Consumer to Corporate).

In [10]:
dim_customer_base = (
    df[['Customer ID', 'Customer Name', 'Segment']]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Distinct customers: {dim_customer_base.shape[0]}")
dim_customer_base.head()

Distinct customers: 793


,Customer ID,Customer Name,Segment
0,CG-12520,Claire Gute,Consumer
1,DV-13045,Darrin Van Huff,Corporate
2,SO-20335,Sean O'Donnell,Consumer
3,BH-11710,Brosina Hoffman,Consumer
4,AA-10480,Andrew Allen,Consumer


In [11]:
# --- Simulate SCD Type 2 history on Segment for a handful of customers ---
# The real dataset only has one Segment value per customer (confirmed in EDA),
# so we manually construct a plausible history for 3 customers to demonstrate
# how the warehouse tracks a customer's attribute changing over time.

dim_customer = dim_customer_base.copy()
dim_customer['effective_date'] = pd.Timestamp('2015-01-01')
dim_customer['end_date'] = pd.NaT
dim_customer['is_current'] = True

# Pick 3 real customer IDs from the data to simulate a segment change for
sample_customers = dim_customer_base['Customer ID'].drop_duplicates().sample(3, random_state=42).tolist()

scd2_new_rows = []
for cust_id in sample_customers:
    current_row = dim_customer[dim_customer['Customer ID'] == cust_id].iloc[0]
    change_date = pd.Timestamp('2017-06-01')  # simulated date the change took effect

    # 1. Close out the original version
    dim_customer.loc[dim_customer['Customer ID'] == cust_id, 'end_date'] = change_date - pd.Timedelta(days=1)
    dim_customer.loc[dim_customer['Customer ID'] == cust_id, 'is_current'] = False

    # 2. Create the new, current version with a different Segment
    other_segments = [s for s in dim_customer_base['Segment'].unique() if s != current_row['Segment']]
    new_segment = other_segments[0]

    scd2_new_rows.append({
        'Customer ID': cust_id,
        'Customer Name': current_row['Customer Name'],
        'Segment': new_segment,
        'effective_date': change_date,
        'end_date': pd.NaT,
        'is_current': True,
    })

dim_customer = pd.concat([dim_customer, pd.DataFrame(scd2_new_rows)], ignore_index=True)
dim_customer['customer_key'] = dim_customer.index + 1  # surrogate PK, one per version

dim_customer[dim_customer['Customer ID'].isin(sample_customers)].sort_values(['Customer ID', 'effective_date'])

,Customer ID,Customer Name,Segment,effective_date,end_date,is_current,customer_key
137,DJ-13510,Don Jones,Corporate,2015-01-01,2017-05-31,False,138
793,DJ-13510,Don Jones,Consumer,2017-06-01,NaT,True,794
198,MD-17350,Maribeth Dona,Consumer,2015-01-01,2017-05-31,False,199
794,MD-17350,Maribeth Dona,Corporate,2017-06-01,NaT,True,795
739,NF-18475,Neil Französisch,Home Office,2015-01-01,2017-05-31,False,740
795,NF-18475,Neil Französisch,Consumer,2017-06-01,NaT,True,796


**Note:** each row in `dim_customer` is now a *version* of a customer, not one row per customer — that's the defining feature of SCD Type 2. The 3 simulated customers now have 2 rows each (old segment + new segment), while everyone else still has exactly 1 row. When building `fact_sales`, an order line must be matched to the correct `customer_key` by checking which version's `effective_date`/`end_date` range the order's date falls into — not just matched on `Customer ID` directly.

## Build `dim_location`

City/State/Postal Code/Region are order-level shipping attributes (confirmed in EDA — 779/793 customers had multiple cities), so they get their own dimension, separate from `dim_customer`.

In [12]:
dim_location = (
    df[['City', 'State', 'Region', 'Postal Code']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_location['location_key'] = dim_location.index + 1  # surrogate PK

print(f"Distinct locations: {dim_location.shape[0]}")
dim_location.head()

Distinct locations: 628


,City,State,Region,Postal Code,location_key
0,Henderson,Kentucky,South,42420,1
1,Los Angeles,California,West,90036,2
2,Fort Lauderdale,Florida,South,33311,3
3,Los Angeles,California,West,90032,4
4,Concord,North Carolina,South,28027,5


In [13]:
df = df.merge(
    dim_location[['City', 'State', 'Region', 'Postal Code', 'location_key']],
    on=['City', 'State', 'Region', 'Postal Code'],
    how='left'
)

assert df['location_key'].isnull().sum() == 0, "Some rows failed to match a location_key"
print(f"dim_location rows: {dim_location.shape[0]}")
print(f"df rows with location_key assigned: {df['location_key'].notnull().sum()} / {df.shape[0]}")

dim_location rows: 628
df rows with location_key assigned: 9800 / 9800


## Build `dim_date`

A generated calendar table covering the full order date range found in EDA (2015-01-03 to 2018-12-30), padded out to cover the latest Ship Date (2019-01-05) so every shipment date also has a matching row.

In [14]:
date_range = pd.date_range(start='2015-01-01', end='2019-01-31', freq='D')

dim_date = pd.DataFrame({'full_date': date_range})
dim_date['date_key'] = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)  # e.g. 20150103
dim_date['day'] = dim_date['full_date'].dt.day
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['month_name'] = dim_date['full_date'].dt.month_name()
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['weekday_name'] = dim_date['full_date'].dt.day_name()
dim_date['is_weekend'] = dim_date['full_date'].dt.dayofweek >= 5

dim_date.head()

,full_date,date_key,day,month,month_name,quarter,year,weekday_name,is_weekend
0,2015-01-01,20150101,1,1,January,1,2015,Thursday,False
1,2015-01-02,20150102,2,1,January,1,2015,Friday,False
2,2015-01-03,20150103,3,1,January,1,2015,Saturday,True
3,2015-01-04,20150104,4,1,January,1,2015,Sunday,True
4,2015-01-05,20150105,5,1,January,1,2015,Monday,False


In [15]:
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True)

df['order_date_key'] = df['Order Date'].dt.strftime('%Y%m%d').astype(int)
df['ship_date_key'] = df['Ship Date'].dt.strftime('%Y%m%d').astype(int)

assert df['order_date_key'].isin(dim_date['date_key']).all(), "Some order dates fall outside dim_date range"
assert df['ship_date_key'].isin(dim_date['date_key']).all(), "Some ship dates fall outside dim_date range"
print("All order and ship dates matched to dim_date.")

All order and ship dates matched to dim_date.


## Save dimension tables to `data/processed/`

These are the files `etl/03_build_fact.py` (and later, `04_load_to_postgres.py`) will read from.

In [16]:
dim_product.to_csv('../data/processed/dim_product.csv', index=False)
dim_customer.to_csv('../data/processed/dim_customer.csv', index=False)
dim_location.to_csv('../data/processed/dim_location.csv', index=False)
dim_date.to_csv('../data/processed/dim_date.csv', index=False)

# Also save df with all surrogate keys attached — this is the input
# 03_build_fact.py will use to construct fact_sales (still needs customer_key
# resolved against the SCD2 date ranges, which happens in that script).
df.to_csv('../data/processed/orders_with_keys.csv', index=False)

print("Saved: dim_product, dim_customer, dim_location, dim_date, orders_with_keys")

Saved: dim_product, dim_customer, dim_location, dim_date, orders_with_keys
